# **Baseline + H2O (GPU-only)**

Cells copied unchanged (outputs kept) from `archive/OS_MAIN.ipynb`: original cell indices [0, 1, 2, 3, 4, 12, 31, 13, 14].

In [1]:
import torch
import time
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, DynamicCache
import transformers

transformers.utils.logging.disable_progress_bar()

torch.cuda.set_per_process_memory_fraction(0.45)

# model_name = "gpt2"
model_name = "facebook/opt-6.7b"

device = "cuda"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    attn_implementation="eager"
).to(device)
model.eval()

print(f"Model: {model_name}")
print(f"Device: {torch.cuda.get_device_name()}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")

Model: facebook/opt-6.7b
Device: NVIDIA A100-SXM4-40GB
VRAM: 39.5 GB


In [2]:
from datasets import load_dataset

# WikiText-2
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")

# Get long articles by filtering for substantial text
long_prompts = []
for item in dataset:
    text = item["text"].strip()
    tokens = tokenizer.encode(text)
    if len(tokens) > 200:  # only keep long articles
        long_prompts.append(text)
    if len(long_prompts) >= 32:
        break

print(f"Found {len(long_prompts)}  prompts")
for i, p in enumerate(long_prompts[:4]):
    tokens = tokenizer.encode(p)
    print(f"Prompt {i}: {len(tokens)} tokens | {p[:80]}...")

Found 32  prompts
Prompt 0: 217 tokens | He had a recurring role in 2003 on two episodes of The Bill , as character " Con...
Prompt 1: 205 tokens | In 2006 Boulter starred in the play Citizenship written by Mark Ravenhill . The ...
Prompt 2: 246 tokens | Boulter starred in two films in 2008 , Daylight Robbery by filmmaker Paris Leont...
Prompt 3: 268 tokens | Du Fu 's work is notable above all for its range . Chinese critics traditionally...


# **BASELINE - Everything on GPU**

In [3]:
def run_baseline_benchmark():

  microbatches_count = [8,16,32]
  n_tokens = [256,512]
  # microbatches_count = [2]
  # n_tokens = [32]
  results = []

  total_vram_mb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 2)

  for D in microbatches_count:
    for N in n_tokens:

      prompts = long_prompts[:D] # Select D prompts

      torch.cuda.empty_cache()
      torch.cuda.reset_peak_memory_stats()

      total_start= time.perf_counter()
      with torch.no_grad():
        kv_caches = []
        next_tokens = []

        # PREFILL
        for i in range(D):
          inputs = tokenizer(prompts[i], return_tensors="pt").to(device)
          outputs = model(input_ids = inputs.input_ids,use_cache = True)
          kv_caches.append(outputs.past_key_values)
          next_token = torch.argmax(outputs.logits[:,-1,:],dim=1).unsqueeze(1)
          next_tokens.append(next_token)


        torch.cuda.reset_peak_memory_stats()
        # DECODE
        for step in range(N-1):
          for i in range(D):
            outputs = model(
                input_ids = next_tokens[i],
                past_key_values = kv_caches[i],
                use_cache = True
              )
            kv_caches[i] = outputs.past_key_values
            next_tokens[i] = torch.argmax(outputs.logits[:,-1,:],dim=1).unsqueeze(1)

      total_end = time.perf_counter()

      # METRICS
      latency = total_end - total_start
      total_tokens = D * N
      throughput = total_tokens / latency
      peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
      vram_util = (peak_vram_mb / total_vram_mb) * 100

      # KV cache size (sum across all D microbatches)
      total_kv_bytes = 0
      for kv in kv_caches:
        for layer in kv.layers:
          total_kv_bytes += layer.keys.nelement() * layer.keys.element_size()
          total_kv_bytes += layer.values.nelement() * layer.values.element_size()
      kv_cache_mb = total_kv_bytes / (1024 ** 2)

      print(f"D={D:<4} N={N:<4} | Latency={latency:.3f}s | Throughput={throughput:.1f} tok/s | Peak VRAM={peak_vram_mb:.1f}MB | KV Cache={kv_cache_mb:.1f}MB | VRAM%={vram_util:.1f}%")

      results.append({
                "Microbatches (D)": D,
                "Tokens Generated (N)": N,
                "Latency (s)": latency,
                "Throughput (tok/s)": throughput,
                "Peak VRAM (MB)": peak_vram_mb,
                "KV Cache Total (MB)": kv_cache_mb,
                "VRAM Util (%)": vram_util,
            })

      # CLEANUP
      del kv_caches, next_tokens
      torch.cuda.empty_cache()

  return pd.DataFrame(results)

print("Baseline benchmark function ready!")

Baseline benchmark function ready!


In [4]:
run_baseline_benchmark()

D=8    N=256  | Latency=55.842s | Throughput=36.7 tok/s | Peak VRAM=14835.1MB | KV Cache=2053.5MB | VRAM%=36.7%
D=8    N=512  | Latency=110.955s | Throughput=36.9 tok/s | Peak VRAM=16116.6MB | KV Cache=3077.5MB | VRAM%=39.9%
D=16   N=256  | Latency=111.975s | Throughput=36.6 tok/s | Peak VRAM=17314.8MB | KV Cache=4091.5MB | VRAM%=42.8%


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 39.49 GiB of which 21.23 GiB is free. Including non-PyTorch memory, this process has 18.25 GiB memory in use. 17.77 GiB allowed; Of the allocated memory 17.14 GiB is allocated by PyTorch, and 628.35 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

# **H2O Baseline - KV Cache Eviction**

In [ ]:
def unpack_kv_cache(kv_cache):
    """Convert DynamicCache or other cache formats into standard ((k,v),(k,v),...) format."""
    if kv_cache is None:
        return None
    # Newest transformers: DynamicCache has .layers list of DynamicLayer objects
    if hasattr(kv_cache, "layers") and isinstance(kv_cache.layers, list):
        return [(layer.keys, layer.values) for layer in kv_cache.layers]
    # Legacy tuple format
    if isinstance(kv_cache, (tuple, list)):
        return kv_cache
    raise TypeError(f"Unsupported KV cache type: {type(kv_cache)}")


def get_seq_len(kv_cache):
    """Return current sequence length of KV cache."""
    layers = unpack_kv_cache(kv_cache)
    if layers is None or len(layers) == 0:
        return 0
    return layers[0][0].shape[-2]


def kv_cache_size_mb(kv_cache):
    """Compute total KV cache memory size in MB."""
    layers = unpack_kv_cache(kv_cache)
    if layers is None:
        return 0
    total = 0
    for k, v in layers:
        total += k.nelement() * k.element_size()
        total += v.nelement() * v.element_size()
    return total / (1024 ** 2)


def offload_kv_to_cpu(kv_cache):
    """Move KV cache to pinned CPU memory for async transfer."""
    layers = unpack_kv_cache(kv_cache)
    return tuple(
        (k.detach().contiguous().cpu().pin_memory(),
         v.detach().contiguous().cpu().pin_memory())
        for k, v in layers
    )


def extract_new_kv_chunk(full_kv_cache, old_seq_len):
    """Extract only newly generated KV entries and store in pinned CPU memory."""
    layers = unpack_kv_cache(full_kv_cache)
    return tuple(
        (k[:, :, old_seq_len:, :].detach().contiguous().cpu().pin_memory(),
         v[:, :, old_seq_len:, :].detach().contiguous().cpu().pin_memory())
        for k, v in layers
    )


def append_kv_chunk(cpu_kv_cache, new_chunk):
    """Append new KV chunk to CPU-side KV cache, keeping result in pinned memory."""
    return tuple(
        (torch.cat([old_k, new_k], dim=-2).contiguous().pin_memory(),
         torch.cat([old_v, new_v], dim=-2).contiguous().pin_memory())
        for (old_k, old_v), (new_k, new_v) in zip(cpu_kv_cache, new_chunk)
    )


def build_kv_cache(kv_layers):
    """Reconstruct a DynamicCache from a list of (key, value) tensor pairs."""
    cache = DynamicCache()
    for layer_idx, (k, v) in enumerate(kv_layers):
        cache.update(k, v, layer_idx)
    return cache

def compute_importance_scores(attentions):
    num_layers = len(attentions)
    num_heads = attentions[0].shape[1]
    num_queries = attentions[0].shape[2]
    num_keys = attentions[0].shape[3]

    scores = torch.zeros(num_keys).to(device)

    for n in range(num_layers):
        layer_attn = attentions[n][0]
        avg_across_heads = layer_attn.sum(dim=0) / num_heads
        avg_across_queries = avg_across_heads.sum(dim=0) / num_queries

        scores += avg_across_queries

    scores = scores / num_layers

    return scores


def classify_indices(top,middle,bottom,scores):
  # sorted scores must be passed here
  sorted_indices = scores.argsort(descending=True)
  top_length = int((len(sorted_indices)/100) * top)
  middle_length = int(((len(sorted_indices)/100)) * middle)
  bottom_lenght = int(((len(sorted_indices)/100)) * bottom)

  hot = sorted_indices[ :top_length]
  cold = sorted_indices[top_length:top_length+middle_length]
  evict = sorted_indices[top_length+middle_length: ]

  return hot,cold,evict

def split_cache(kv_cache, hot_indices, cold_indices):
    hot_layers = []
    cold_layers = []

    for k, v in kv_cache:
        h_idx = hot_indices.to(k.device)
        c_idx = cold_indices.to(k.device)

        hot_k = k[:, :, h_idx, :]
        hot_v = v[:, :, h_idx, :]
        hot_layers.append((hot_k, hot_v))

        cold_k = k[:, :, c_idx, :]
        cold_v = v[:, :, c_idx, :]
        cold_layers.append((cold_k, cold_v))

    return hot_layers, cold_layers

In [3]:
def run_h2o_benchmark(budget_ratio=0.4, recent_ratio=0.5):
    """H2O eviction on GPU only — no CPU offloading, no prefetching."""
    microbatches_count = [8, 16,32]
    n_tokens = [256, 512]
    results = []

    total_vram_mb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 2)

    for D in microbatches_count:
        for N in n_tokens:

            prompts = long_prompts[:D]
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

            sample_len = tokenizer(prompts[0], return_tensors="pt").input_ids.shape[1]
            budget = int((sample_len + N) * budget_ratio)
            budget = max(budget, sample_len)
            recent_window = int(budget * recent_ratio)

            total_start = time.perf_counter()

            with torch.no_grad():

                # PREFILL
                next_tokens = []
                cum_scores_list = []
                kv_caches = []
                for i in range(D):
                    inputs = tokenizer(prompts[i], return_tensors="pt").to(device)
                    outputs = model(input_ids=inputs.input_ids, use_cache=True, output_attentions=True)
                    next_tokens.append(torch.argmax(outputs.logits[:, -1, :], dim=1).unsqueeze(1))

                    seq_len = outputs.past_key_values.get_seq_length()
                    scores = torch.zeros(seq_len, device=device)
                    for attn in outputs.attentions:
                        scores += attn[0].sum(dim=0).sum(dim=0)
                    cum_scores_list.append(scores)
                    kv_caches.append(outputs.past_key_values)
                    # H2O keeps caches on GPU, can't offload here

                torch.cuda.reset_peak_memory_stats()

                # DECODE
                for step in range(N - 1):
                    for i in range(D):

                        outputs = model(
                            input_ids=next_tokens[i],
                            past_key_values=kv_caches[i],
                            use_cache=True,
                            output_attentions=True
                        )
                        kv_caches[i] = outputs.past_key_values
                        next_tokens[i] = torch.argmax(outputs.logits[:, -1, :], dim=1).unsqueeze(1)

                        # Update cumulative scores
                        current_len = kv_caches[i].get_seq_length()
                        new_score = torch.zeros(1, device=device)
                        cum_scores_list[i] = torch.cat([cum_scores_list[i], new_score])
                        for attn in outputs.attentions:
                            step_scores = attn[0].sum(dim=0).sum(dim=0)
                            if len(step_scores) == len(cum_scores_list[i]):
                                cum_scores_list[i] += step_scores

                        # H2O eviction
                        while current_len > budget:
                            evict_candidates = cum_scores_list[i][:-recent_window].clone()
                            if len(evict_candidates) == 0:
                                break

                            evict_pos = evict_candidates.argmin().item()

                            keep_mask = torch.ones(current_len, dtype=torch.bool, device=device)
                            keep_mask[evict_pos] = False
                            keep_indices = torch.where(keep_mask)[0]

                            kv_layers = unpack_kv_cache(kv_caches[i])
                            kept_layers = [(k[:, :, keep_indices, :], v[:, :, keep_indices, :]) for k, v in kv_layers]
                            kv_caches[i] = build_kv_cache(kept_layers)

                            cum_scores_list[i] = cum_scores_list[i][keep_mask]
                            current_len -= 1

                        del outputs

            total_end = time.perf_counter()

            latency = total_end - total_start
            throughput = (D * N) / latency
            peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
            vram_util = (peak_vram_mb / total_vram_mb) * 100

            kv_bytes = 0
            for cache in kv_caches:
                kv_layers = unpack_kv_cache(cache)
                for k, v in kv_layers:
                    kv_bytes += k.nelement() * k.element_size()
                    kv_bytes += v.nelement() * v.element_size()
            kv_mb = kv_bytes / (1024 ** 2)

            print(f"D={D:<4} N={N:<4} | Budget={budget:<4} | Latency={latency:.3f}s | Throughput={throughput:.1f} tok/s | "
                  f"Peak VRAM={peak_vram_mb:.1f}MB | VRAM%={vram_util:.1f}% | GPU KV={kv_mb:.1f}MB")

            results.append({
                "Microbatches (D)": D,
                "Tokens Generated (N)": N,
                "Budget": budget,
                "Latency (s)": latency,
                "Throughput (tok/s)": throughput,
                "Peak VRAM (MB)": peak_vram_mb,
                "VRAM Util (%)": vram_util,
                "GPU KV (MB)": kv_mb,
            })

            del kv_caches, next_tokens, cum_scores_list
            torch.cuda.empty_cache()

    return pd.DataFrame(results)

In [6]:
run_h2o_benchmark()

D=8    N=256  | Budget=217  | Latency=66.340s | Throughput=30.9 tok/s | Peak VRAM=15421.4MB | VRAM%=38.1% | GPU KV=868.0MB
D=8    N=512  | Budget=291  | Latency=129.027s | Throughput=31.7 tok/s | Peak VRAM=15597.1MB | VRAM%=38.6% | GPU KV=1164.0MB
D=16   N=256  | Budget=217  | Latency=131.977s | Throughput=31.0 tok/s | Peak VRAM=16733.3MB | VRAM%=41.4% | GPU KV=1736.0MB
D=16   N=512  | Budget=291  | Latency=258.209s | Throughput=31.7 tok/s | Peak VRAM=16759.5MB | VRAM%=41.4% | GPU KV=2328.0MB


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 39.49 GiB of which 21.22 GiB is free. Including non-PyTorch memory, this process has 18.26 GiB memory in use. 17.77 GiB allowed; Of the allocated memory 17.35 GiB is allocated by PyTorch, and 416.27 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)